# Parallelizing with JAX
In a previous notebook, we've explored how to get started with JAX on TIKE. We've seen already that we can dramatically accelerate our code with even relatively simple changes to our code.

One of the benefits of using TIKE is that we have access to multiple cores. How can we use JAX to most efficiently use all these cores? Let's find out and make sure we're *parallelizing* correctly!

First, let's again make sure JAX is installed in this kernel.

In [1]:
!pip install -U jax

In [2]:
import jax
import jax.numpy as jnp
import numpy as np

# Implicit parallelization with JAX

To explore how JAX parallelizes things, let's first instantiate a large array.

As you run these computations, watch the "kernel usage" in the tab to the right of your screen, in particular the "Host CPU".

In [3]:
big_array = jnp.ones((10000, 10000))

Let's dot this array with itself, timing how long the operation takes.

In [7]:
%%time
jnp.dot(big_array, big_array).block_until_ready()

CPU times: user 26.2 s, sys: 1.06 s, total: 27.2 s
Wall time: 7.91 s


Array([[10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       ...,
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.]],      dtype=float32)

Let's do the same with a `numpy` array.

In [5]:
big_array = np.ones((10000, 10000))

In [53]:
%%time
np.dot(big_array, big_array)

CPU times: user 49.8 s, sys: 2.21 s, total: 52 s
Wall time: 13.9 s


array([[10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       ...,
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.],
       [10000., 10000., 10000., ..., 10000., 10000., 10000.]])

In both cases, we were able to use 100% of our CPUs. This seems a bit like magic — we didn't even explictly request any parallelization! Note, of course, that the jax implementation was faster. This is because JAX is friendlier to the device used.

It turns out, both `jax` and vanilla `numpy` do perform some parallelization on array operations under the hood. `Numpy` operations such as `np.dot` utilize calls to the BLAS routines, which are essentially a large number of linear algebra routines implemented in Fortran and C. BLAS automatically recognizes the number of cores available in your current system and will create multiple *threads* to take advantage of the these cores.

`JAX` operates similarly, with *its* linear algebra engine (XLA) making use of the available local cores, as well.

# Vectorization with JAX
There's another clear way in which we can take advantage of your hardware to speed up code. That's through something called *vectorization*.

Essentially, vectorization eliminates the inefficiencies of explicit loops. Modern computer hardware often has the ability to perform the same operation multiple times on an array, so long as it is prompted to do so. Normally, loops don't take advantage of this, while vectorized code can. 

Writing vectorized code can be tricky, though. It requires keeping strict track of, for instance, array dimensions.

Luckily, JAX has a feature known as "autovectorization." Because JAX has the ability to trace through your Python functions and fuse operations together, it can automatically recognize which portions of your code can be vectorized.

Let's walk through how to do this. First, let's write a function that operates on two arrays. It takes all the values of the first array and multiplies them by the maximum of the second array.

In [20]:
def array_op(array1, array2):
    max_val = jnp.max(array2)
    return array1 * max_val

Let's make a few arrays now!

In [21]:
array1 = jnp.arange(25)
array2 = jnp.linspace(-29, 20, 4)

Now, let's time our operation.

In [29]:
%%timeit
array_op(array1, array2).block_until_ready()

13.6 μs ± 263 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


Now, let's stack some arrays and manually loop over them to perform our calculation. Note that this is the inefficient way to do it!

In [30]:
array1s = jnp.stack([array1, array1, array1])
array2s = jnp.stack([array2, array2, array2])

In [31]:
%%timeit
for array1, array2 in zip(array1s, array2s):
    array_op(array1, array2).block_until_ready()

60.3 μs ± 1.4 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Now, let's vectorize instead.

In [34]:
auto_array_op = jax.vmap(array_op)

In [35]:
%%timeit
auto_array_op(array1s, array2s).block_until_ready()

442 μs ± 10.3 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


What gives? This is slower than before!

The reason is that `JAX`'s `vmap` performs well when the batch dimension is large. Let's try with a larger dimension.

In [47]:
batch_size = 1000
array1s = jnp.stack([array1] * batch_size)   # shape: (1000, 25)
array2s = jnp.stack([array2] * batch_size)   # shape: (1000, 4)

In [48]:
%%timeit
for array1, array2 in zip(array1s, array2s):
    array_op(array1, array2).block_until_ready()

24.2 ms ± 1.67 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [49]:
%%timeit
auto_array_op(array1s, array2s).block_until_ready()

494 μs ± 15.2 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Wow! The vectorized version scales very well.

We can also compose `JAX` functions together. Recall that `jit` can speed things up quite a bit; let's try that.

In [50]:
jitted_array_op = jax.jit(auto_array_op)

jitted_array_op(array1s, array2s)

Array([[  0.,  20.,  40., ..., 440., 460., 480.],
       [  0.,  20.,  40., ..., 440., 460., 480.],
       [  0.,  20.,  40., ..., 440., 460., 480.],
       ...,
       [  0.,  20.,  40., ..., 440., 460., 480.],
       [  0.,  20.,  40., ..., 440., 460., 480.],
       [  0.,  20.,  40., ..., 440., 460., 480.]], dtype=float32)

In [52]:
%%timeit
jitted_array_op(array1s, array2s).block_until_ready()

30.1 μs ± 814 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


This is almost 3 orders of magnitude faster than the non-vectorized version — with minimal changes to our code!

# When parallelizing with JAX does not work as well

We've just shown a few ways in which parallelization with JAX requires (relatively) minor changes to our code. This is in part because we've hand-picked a few suitable examples! There are certainly cases in which parallelization with JAX is not as useful.

For instance: if your computation is very string-heavy (e.g., concatenating large strings), JAX is less useful, because it cannot vectorize string operations. Similarly, for sequential I/O type operations or for functions with "side effects" (e.g., writing logs), these types of operations cannot be performed in parallel. Finally, if your function cannot be traced by JAX (e.g., it includes irregular Python classes), JAX can't speed it up.

In more straightforward cases, though — numeric calculations on dense, numeric, fixed-shape arrays, especially larger ones — JAX should be able to help.